# KOF Governors con `source_url`: Notebook explicado

Este notebook procesa la base `cbg_turnover_v23upload.xlsx` del KOF ETH Zurich y genera un CSV limpio con una URL oficial por registro.

## Output final

- `iso3`
- `country`
- `name`
- `position`
- `start_year`
- `end_year`
- `status`
- `source_url`


## 1. Imports, configuración y mapa de fuentes

Este notebook queda **autocontenido**: incluye el mapa de `source_url` dentro de la misma libreta, sin depender de archivos auxiliares externos.


In [ ]:
import csv
import re
from pathlib import Path

import openpyxl
import pandas as pd

INPUT_FILE = Path("data/cbg_turnover_v23upload.xlsx")
SHEET_NAME = "governors v2023"
OUTPUT_FILE = Path("data/kof_governors_with_sources.csv")
UNRESOLVED_OUTPUT_FILE = Path("data/kof_missing_source_url.csv")

# Si quieres detener el notebook cuando falten source_url, cambia esto a True.
STRICT_SOURCE_URL = False

# Alias manuales para corregir ISO3 raros o legacy del archivo KOF.
ISO_ALIASES = {
    "AOA": "AGO",  # Angola
    "ROM": "ROU",  # Romania legacy
    "BUR": "MMR",  # Burma/Myanmar legacy
    "ZAR": "COD",  # Zaire -> DR Congo
    "SER": "SRB",  # Serbia legacy
}

# URL oficial o institucional por entidad/ISO3.
SOURCE_URLS = {'AFG': 'https://dab.gov.af', 'ALB': 'https://www.bankofalbania.org', 'DZA': 'https://www.bank-of-algeria.dz', 'AGO': 'https://www.bna.ao', 'ARG': 'https://www.bcra.gob.ar', 'ARM': 'https://www.cba.am', 'ABW': 'https://www.cba.aw', 'AUS': 'https://www.rba.gov.au', 'AUT': 'https://www.oenb.at', 'AZE': 'https://www.cbar.az', 'BHS': 'https://www.centralbankbahamas.com', 'BHR': 'https://www.cbb.gov.bh', 'BGD': 'https://www.bb.org.bd', 'BRB': 'https://www.centralbank.org.bb', 'BLR': 'https://www.nbrb.by', 'BEL': 'https://www.nbb.be', 'BLZ': 'https://www.centralbank.org.bz', 'BEN': 'https://www.bceao.int', 'BMU': 'https://www.bma.bm', 'BTN': 'https://www.rma.org.bt', 'BOL': 'https://www.bcb.gob.bo', 'BIH': 'https://www.cbbh.ba', 'BWA': 'https://www.bankofbotswana.bw', 'BRA': 'https://www.bcb.gov.br', 'BRN': 'https://www.ambd.gov.bn', 'BGR': 'https://www.bnb.bg', 'BFA': 'https://www.bceao.int', 'BDI': 'https://www.brb.bi', 'KHM': 'https://www.nbc.gov.kh', 'CMR': 'https://www.beac.int', 'CAN': 'https://www.bankofcanada.ca', 'CAF': 'https://www.beac.int', 'TCD': 'https://www.beac.int', 'CHL': 'https://www.bcentral.cl', 'CHN': 'https://www.pbc.gov.cn', 'COL': 'https://www.banrep.gov.co', 'COM': 'https://www.bcc.km', 'COG': 'https://www.beac.int', 'COD': 'https://www.bcc.cd', 'CRI': 'https://www.bccr.fi.cr', 'HRV': 'https://www.hnb.hr', 'CUB': 'https://www.bc.gob.cu', 'CYP': 'https://www.centralbank.cy', 'CZE': 'https://www.cnb.cz', 'DNK': 'https://www.nationalbanken.dk', 'DJI': 'https://www.banque-centrale.dj', 'DOM': 'https://www.bancentral.gov.do', 'ECU': 'https://www.bce.fin.ec', 'EGY': 'https://www.cbe.org.eg', 'SLV': 'https://www.bcr.gob.sv', 'GNQ': 'https://www.beac.int', 'ERI': 'https://www.bankoferitreanot.er', 'EST': 'https://www.eestipank.ee', 'SWZ': 'https://www.centralbank.org.sz', 'ETH': 'https://nbe.gov.et', 'FJI': 'https://www.rbf.gov.fj', 'FIN': 'https://www.suomenpankki.fi', 'FRA': 'https://www.banque-france.fr', 'GAB': 'https://www.beac.int', 'GMB': 'https://www.cbg.gm', 'GEO': 'https://www.nbg.gov.ge', 'DEU': 'https://www.bundesbank.de', 'GHA': 'https://www.bog.gov.gh', 'GRC': 'https://www.bankofgreece.gr', 'GTM': 'https://www.banguat.gob.gt', 'GIN': 'https://www.bcrg-guinee.org', 'GNB': 'https://www.bceao.int', 'GUY': 'https://www.bankofguyana.org.gy', 'HTI': 'https://www.brh.net', 'HND': 'https://www.bch.hn', 'HKG': 'https://www.hkma.gov.hk', 'HUN': 'https://www.mnb.hu', 'ISL': 'https://www.cb.is', 'IND': 'https://www.rbi.org.in', 'IDN': 'https://www.bi.go.id', 'IRN': 'https://www.cbi.ir', 'IRQ': 'https://www.cbi.iq', 'IRL': 'https://www.centralbank.ie', 'ISR': 'https://www.boi.org.il', 'ITA': 'https://www.bancaditalia.it', 'JAM': 'https://www.boj.org.jm', 'JPN': 'https://www.boj.or.jp', 'JOR': 'https://www.cbj.gov.jo', 'KAZ': 'https://www.nationalbank.kz', 'KEN': 'https://www.centralbank.go.ke', 'PRK': 'https://www.kcna.kp', 'KOR': 'https://www.bok.or.kr', 'KWT': 'https://www.cbk.gov.kw', 'KGZ': 'https://www.nbkr.kg', 'LAO': 'https://www.bol.gov.la', 'LVA': 'https://www.bank.lv', 'LBN': 'https://www.bdl.gov.lb', 'LSO': 'https://www.centralbank.org.ls', 'LBR': 'https://www.cbl.org.lr', 'LBY': 'https://cbl.gov.ly', 'LTU': 'https://www.lb.lt', 'LUX': 'https://www.bcl.lu', 'MAC': 'https://www.amcm.gov.mo', 'MDG': "https://www.banky-foiben'ny-madagasikara.mg", 'MWI': 'https://www.rbm.mw', 'MYS': 'https://www.bnm.gov.my', 'MDV': 'https://www.mma.gov.mv', 'MLI': 'https://www.bceao.int', 'MLT': 'https://www.centralbankmalta.org', 'MRT': 'https://www.bcm.mr', 'MUS': 'https://www.bom.mu', 'MEX': 'https://www.banxico.org.mx', 'MDA': 'https://www.bnm.md', 'MNG': 'https://www.mongolbank.mn', 'MNE': 'https://www.cbcg.me', 'MAR': 'https://www.bkam.ma', 'MOZ': 'https://www.bancomoc.mz', 'MMR': 'https://www.cbm.gov.mm', 'NAM': 'https://www.bon.com.na', 'NPL': 'https://www.nrb.org.np', 'NLD': 'https://www.dnb.nl', 'NZL': 'https://www.rbnz.govt.nz', 'NIC': 'https://www.bcn.gob.ni', 'NER': 'https://www.bceao.int', 'NGA': 'https://www.cbn.gov.ng', 'MKD': 'https://www.nbrm.mk', 'NOR': 'https://www.norges-bank.no', 'OMN': 'https://www.cbo.gov.om', 'PAK': 'https://www.sbp.org.pk', 'PSE': 'https://www.pma.ps', 'PAN': 'https://www.superbancos.gob.pa', 'PNG': 'https://www.bankpng.gov.pg', 'PRY': 'https://www.bcp.gov.py', 'PER': 'https://www.bcrp.gob.pe', 'PHL': 'https://www.bsp.gov.ph', 'POL': 'https://www.nbp.pl', 'PRT': 'https://www.bportugal.pt', 'QAT': 'https://www.qcb.gov.qa', 'ROU': 'https://www.bnr.ro', 'RUS': 'https://www.cbr.ru', 'RWA': 'https://www.bnr.rw', 'SAU': 'https://www.sama.gov.sa', 'SEN': 'https://www.bceao.int', 'SRB': 'https://www.nbs.rs', 'SYC': 'https://www.cbs.sc', 'SLE': 'https://www.bsl.gov.sl', 'SGP': 'https://www.mas.gov.sg', 'SVK': 'https://www.nbs.sk', 'SVN': 'https://www.bsi.si', 'SLB': 'https://www.cbsi.com.sb', 'SOM': 'https://www.centralbank.gov.so', 'ZAF': 'https://www.resbank.co.za', 'SSD': 'https://www.bss.gov.ss', 'ESP': 'https://www.bde.es', 'LKA': 'https://www.cbsl.gov.lk', 'SDN': 'https://www.cbos.gov.sd', 'SUR': 'https://www.cbvs.sr', 'SWE': 'https://www.riksbank.se', 'CHE': 'https://www.snb.ch', 'SYR': 'https://www.cb.gov.sy', 'TWN': 'https://www.cbc.gov.tw', 'TJK': 'https://www.nbt.tj', 'TZA': 'https://www.bot.go.tz', 'THA': 'https://www.bot.or.th', 'TLS': 'https://www.bancocentral.tl', 'TGO': 'https://www.bceao.int', 'TON': 'https://www.nrbt.to', 'TTO': 'https://www.central-bank.org.tt', 'TUN': 'https://www.bct.gov.tn', 'TUR': 'https://www.tcmb.gov.tr', 'TKM': 'https://www.cbt.gov.tm', 'UGA': 'https://www.bou.or.ug', 'UKR': 'https://www.bank.gov.ua', 'ARE': 'https://www.centralbank.ae', 'GBR': 'https://www.bankofengland.co.uk', 'USA': 'https://www.federalreserve.gov', 'URY': 'https://www.bcu.gub.uy', 'UZB': 'https://www.cbu.uz', 'VUT': 'https://www.rbv.gov.vu', 'VEN': 'https://www.bcv.org.ve', 'VNM': 'https://www.sbv.gov.vn', 'YEM': 'https://www.centralbank.gov.ye', 'ZMB': 'https://www.boz.zm', 'ZWE': 'https://www.rbz.co.zw', 'CPV': 'https://www.bcv.cv', 'PYF': 'https://www.ieom.fr/polynesie-francaise/', 'KIR': 'https://www.kfsa.gov.ki/', 'STP': 'https://www.bcstp.st', 'WSM': 'https://www.cbs.gov.ws', 'EUR': 'https://www.ecb.europa.eu', 'ECB': 'https://www.ecb.europa.eu', 'XAF': 'https://www.beac.int', 'BEAC': 'https://www.beac.int/beac/le-gouvernement-de-la-banque/', 'XOF': 'https://www.bceao.int', 'BCEAO': 'https://www.bceao.int/en/content/governor', 'XCD': 'https://www.eccb-centralbank.org', 'ECCB': 'https://www.eccb-centralbank.org/about-us/governance', 'ANT': 'https://www.centralbank.cw'}


## 2. Funciones auxiliares

Aquí están las funciones que normalizan ISO3, limpian nombres, extraen años y parsean cada celda de la hoja.

In [ ]:
def normalize_iso(iso_value):
    iso = str(iso_value or "").strip().upper()
    return ISO_ALIASES.get(iso, iso)


def extract_year(s):
    if not s:
        return ""
    s = str(s).lower().strip()
    if any(x in s for x in ["present", "current", "ongoing"]):
        return "En cargo"
    m = re.search(r"\d{4}", s)
    return m.group() if m else ""


def clean_name(name):
    name = str(name or "")
    name = re.sub(
        r"^(dr\.|mr\.|mrs\.|ms\.|prof\.|ing\.|sir\s)\s*",
        "",
        name,
        flags=re.IGNORECASE,
    )
    # Quitar notas editoriales y de reelección/mandato en paréntesis.
    name = re.sub(
        r"\s*\((?:[^)]*(?:reapp|reappointed|interim|acting|first|second|third|fourth|1st|2nd|3rd|4th|term|time)[^)]*)\)",
        "",
        name,
        flags=re.IGNORECASE,
    )
    name = re.sub(r"^\*+\s*", "", name)
    name = re.sub(r"\s+", " ", name).strip().rstrip(".,;")
    return name


def parse_cell(text, country_name, iso):
    if not text or str(text).strip() in ["None", ""]:
        return []

    text = str(text).strip()
    skip_keywords = [
        "established", "est. in", "est.in", "code:", "from 1985 onwards member",
        "member of beac", "(beac)", "(bceao)", "(eccb)", "no central bank",
        "code: -999", "exists since", "est. 1", "est.1",
    ]
    if any(p in text.lower() for p in skip_keywords):
        return []

    patterns = [
        r"(\d{1,2}-\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{1,2}-\d{4})\s+(.+)",
        r"(\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{4})\s+(.+)",
        r"(\d{1,2}-\d{4})\s+to\s+(present|current)\s+(.+)",
        r"(\d{4})\s+to\s+(\d{4})\s+(.+)",
        r"(\d{4})\s+to\s+(present|current)\s+(.+)",
        r"(\d{4})\s+-\s+(\d{4})\s+(.+)",
        r"(\d{4})\s+-\s+(present|current)\s+(.+)",
        r"(.+?)\s+(\d{1,2}-\d{4})\s+to\s+(\d{1,2}-\d{4})$",
        r"(.+?)\s+(\d{4})\s+to\s+(\d{4})$",
    ]

    results = []
    for entry in re.split(r"\n", text):
        entry = entry.strip()
        if not entry:
            continue

        for i, pat in enumerate(patterns):
            m = re.search(pat, entry, re.IGNORECASE)
            if not m:
                continue

            if i < 7:
                start_raw, end_raw, name = m.group(1), m.group(2), m.group(3)
            else:
                name, start_raw, end_raw = m.group(1), m.group(2), m.group(3)

            name = clean_name(name)
            if len(name) < 2:
                continue

            start_year = extract_year(start_raw)
            end_year = extract_year(end_raw)
            if not start_year:
                continue

            results.append(
                {
                    "iso3": iso,
                    "country": country_name,
                    "name": name,
                    "position": "President / Governor",
                    "start_year": start_year,
                    "end_year": end_year,
                    "status": "Actual" if end_year == "En cargo" else "Histórico",
                    "source_url": SOURCE_URLS.get(iso, ""),
                }
            )
            break

    return results


## 3. Validar archivo, hoja y mapa de URLs

Aquí comprobamos que el archivo exista, que la hoja esté presente y detectamos los ISO3 que quedarían sin `source_url`.

Si `STRICT_SOURCE_URL = True`, el notebook se detiene.
Si está en `False`, exporta además una tabla de pendientes en `data/kof_missing_source_url.csv`.

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {INPUT_FILE}")

wb = openpyxl.load_workbook(INPUT_FILE, read_only=True, data_only=True)
print("Hojas disponibles:", wb.sheetnames)

if SHEET_NAME not in wb.sheetnames:
    raise ValueError(f"La hoja {SHEET_NAME!r} no existe en el archivo")

ws = wb[SHEET_NAME]
rows = list(ws.iter_rows(values_only=True))
iso_codes = rows[0]
country_names = rows[1]

countries = {}
for i, (iso, name) in enumerate(zip(iso_codes, country_names)):
    if iso and name:
        countries[i] = {
            "iso": normalize_iso(iso),
            "name": str(name).strip(),
        }

country_map_df = pd.DataFrame(countries.values()).drop_duplicates().sort_values(["iso", "name"])
country_map_df["source_url"] = country_map_df["iso"].map(SOURCE_URLS).fillna("")
unresolved_source_df = country_map_df[country_map_df["source_url"] == ""].copy()

print(f"Países encontrados: {len(country_map_df)}")
print(f"Filas de datos: {len(rows) - 2}")
print(f"ISO3 sin source_url: {len(unresolved_source_df)}")
display(unresolved_source_df.head(30))

if len(unresolved_source_df) > 0:
    unresolved_source_df.to_csv(UNRESOLVED_OUTPUT_FILE, index=False)
    print(f"Pendientes guardados en: {UNRESOLVED_OUTPUT_FILE}")
    if STRICT_SOURCE_URL:
        raise ValueError("Hay ISO3 sin source_url. Revisa el archivo de pendientes antes de continuar.")


## 4. Parsear todas las celdas

In [ ]:
all_records = []
for row in rows[2:]:
    for col_idx, val in enumerate(row):
        if col_idx not in countries:
            continue
        if not val or str(val).strip() in ["None", ""]:
            continue
        ci = countries[col_idx]
        all_records.extend(parse_cell(val, ci["name"], ci["iso"]))

print(f"Registros parseados: {len(all_records)}")
all_records[:5]


## 5. Deduplicar mejor

La deduplicación usa `iso3`, `name`, `start_year` y `end_year` para no colapsar casos distintos del mismo nombre.

In [ ]:
seen = set()
unique = []
for r in all_records:
    key = (r["iso3"], r["name"].lower(), r["start_year"], r["end_year"])
    if key not in seen:
        seen.add(key)
        unique.append(r)

print(f"Registros únicos: {len(unique)}")


## 6. Ver resultado procesado

In [ ]:
processed_df = pd.DataFrame(unique)
processed_df = processed_df.sort_values(["country", "start_year", "name"], na_position="last")
print(processed_df.shape)
display(processed_df.head(30))


## 7. Exportar CSV final

In [ ]:
fieldnames = [
    "iso3", "country", "name", "position", "start_year", "end_year",
    "status", "source_url"
]

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(processed_df.to_dict(orient="records"))

print(f"CSV guardado en: {OUTPUT_FILE}")


## 8. Ver el CSV guardado

In [ ]:
saved_df = pd.read_csv(OUTPUT_FILE)
print(saved_df.shape)
display(saved_df.head(30))


## 9. Estadísticas y chequeos relevantes

In [ ]:
print("Resumen general")
print("-" * 40)
print("Total registros        :", len(saved_df))
print("Países únicos          :", saved_df["iso3"].nunique())
print("Gobernadores actuales  :", (saved_df["end_year"] == "En cargo").sum())
print("Gobernadores históricos:", (saved_df["status"] == "Histórico").sum())

start_year_num = pd.to_numeric(saved_df["start_year"], errors="coerce")
end_year_num = pd.to_numeric(saved_df["end_year"], errors="coerce")

print("\nCobertura temporal")
print("-" * 40)
print("Año inicio mínimo      :", int(start_year_num.min()) if start_year_num.notna().any() else None)
print("Año inicio máximo      :", int(start_year_num.max()) if start_year_num.notna().any() else None)
print("Año fin mínimo         :", int(end_year_num.min()) if end_year_num.notna().any() else None)
print("Año fin máximo         :", int(end_year_num.max()) if end_year_num.notna().any() else None)

top_countries = (
    saved_df.groupby(["iso3", "country"])
    .size()
    .reset_index(name="governor_count")
    .sort_values("governor_count", ascending=False)
)
print("\nPaíses con más gobernadores")
print("-" * 40)
display(top_countries.head(15))

current_by_country = (
    saved_df[saved_df["end_year"] == "En cargo"]
    .groupby(["iso3", "country"])
    .size()
    .reset_index(name="current_rows")
    .sort_values("current_rows", ascending=False)
)
multi_current = current_by_country[current_by_country["current_rows"] > 1]
print("Países con más de un actual:", len(multi_current))
if len(multi_current) > 0:
    display(multi_current)

duplicate_names = (
    saved_df.groupby(["iso3", "country", "name"])
    .size()
    .reset_index(name="n")
    .query("n > 1")
    .sort_values("n", ascending=False)
)
print("Nombres repetidos en mismo país:", len(duplicate_names))
if len(duplicate_names) > 0:
    display(duplicate_names.head(20))
